In [ ]:
import torch

print("Torch Version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


In [ ]:
from utils import PROCESSED_DATA_DIR
import pandas as pd
import os

intraday_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "merged.csv"), index_col=0, parse_dates=True)
day_ahead_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "quarterly.csv"), index_col=0, parse_dates=True)

In [ ]:
intraday_df.head(10)

In [ ]:
day_ahead_df.head(10)

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

day_ahead_df["item_id"] = "day_ahead"
day_ahead_df["timestamp"] = day_ahead_df.index.tz_convert(None)
intraday_df["item_id"] = "intraday"
intraday_df["timestamp"] = intraday_df.index.tz_convert(None)

ts_day_ahead = TimeSeriesDataFrame.from_data_frame(day_ahead_df)
ts_intraday = TimeSeriesDataFrame.from_data_frame(intraday_df)

In [ ]:
from utils import MODEL_DIR
from autogluon.timeseries import TimeSeriesPredictor

predictor_da = TimeSeriesPredictor(
    target="DA Price (EUR/MWh)",
    freq="h",
    prediction_length=24,
    eval_metric="MAPE",
    path=os.path.join(MODEL_DIR, "autogluon_da"),
    verbosity=2,
)
predictor_da.fit(ts_day_ahead)

In [ ]:
predictor_id = TimeSeriesPredictor(
    target="ID Weighted average of the prices (EUR/MWh)",
    freq="15min",
    prediction_length=96,
    eval_metric="MAPE",
    path=os.path.join(MODEL_DIR, "autogluon_id"),
    verbosity=2,
)
predictor_id.fit(ts_intraday)

In [ ]:
predictor_da.leaderboard()

In [ ]:
predictor_id.leaderboard()